In [ ]:
# 환경 (MiniCarEnv)
# - 상태: [x, y, θ, v]
#   - 행동: 0=좌, 1=직진, 2=우
#   - 보상: 도로 중심선(y=50)과의 거리 벌점
#           x>90 도달 시 보상 +10, 에피소드 종료
#           도로 이탈 시 감점 -10, 종료,  그 외는 보상 +1
#  DQN 모델 정의
#   - Dense(64) × 2(은닉층) → Linear 출력층
#   - MSE loss, Adam optimizer
#  학습 루프
#   epsilon-greedy 정책
#   experience replay (replay buffer: deque)
#   target network 주기적 동기화

# pip install gymnasium

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import animation
from tensorflow import keras
from collections import deque
import random
import gymnasium as gym
from gymnasium import spaces  # 강화학습 환경 정의시 사용할 상태공간과 행동공간 설정용

In [ ]:
# 환경 : 미니카가 도로를 달리며 목적지에 도달하고 도로 중앙선(y=50)에 가까이 있도록 학습하게끔 설계
class MiniCarEnv(gym.Env):
    def __init__(self):
        super(MiniCarEnv, self).__init__()
        # 상태 공간 : 상태는 [x, y, θ, v]의 4차원 벡터
        # x: 수평위치 (0~100), y: 수직위치 (도로중앙은 y=50), θ:방향 각도(-π ~ +π), v:속도(0~5)
        # 이 공간에서 미니카의 현재 상태를 표현
        self.observation_space = spaces.Box(low=np.array([0, 0, -np.pi, 0], dtype=np.float32), \
                    high = np.array([100, 100, np.pi, 5], dtype=np.float32))
        # 행동은 3가지 : 0(좌), 1(직진), 2(우)
        self.action_space = spaces.Discrete(3)
        self.reset()

    # 에피소드 시작 시 초기 상태를 설정
    def reset(self, seed=None, options=None):
        x = 10.0      # 왼쪽에서 시작
        y = 50.0      # 도로 중앙
        theta = 0.0   # θ 오른쪽 정면
        v = 1.0       # 속도
        self.state = np.array([x, y, theta, v])
        return self.state, {}  # {}는 빈 딕셔너리, Gymnasium의 표준반환형식을 따르기 위해 포함

    # 한 개의 타임 스텝 진행 함수
    def step(self, action):
        # self.state는 NumPy 배열로, 미니카의 현재 상태를 나타냄.
        x, y, theta, v = self.state.astype(np.float64)

        # 조향 업데이트 ---
        steer_step = 0.10
        if action == 0:            # 왼쪽 조향(방향을 조금 조절)
            theta -= steer_step    # theta를 -0.1만큼 줄이면 → 왼쪽으로 약간 꺾음
        elif action == 2:          # 오른쪽 조향
            theta += steer_step    # theta를 +0.1만큼 늘리면 → 오른쪽으로 약간 꺾음

        theta *= 0.98         # 조향 감쇠(자연스럽게 직진으로 돌아오게), 과도한 선회 억제
        theta = (theta + np.pi) % (2 * np.pi) - np.pi # 각도 래핑:[-pi, pi]로 유지(무한히 커지지 않게)
        # 각도를 향상 [-pi, pi] (-180 ~ 180) 범위 안에 맞추기

        # 이동 ---
        n = np.random.normal(0, 0.02, size=2)  # 강화학습에 반영할 노이즈(센서오차, 바람, 미끄러짐 등)
        x_prev = x

        # 조향 조정 후 방향을 따라 이동(theta를 반영해 이동)
        # 물체가 속도 V로 방향 θ를 향해 움직일 때의 x축 이동량
        x = x + v * np.cos(theta) + n[0]

        y = y + v * np.sin(theta) + n[1] # 이 두 줄로 θ방향을 기준으로 한 칸 이동하는 것
        # 예: 처음 theta = 0이면 → 오른쪽(x축 방향)으로 직진,
        # theta += 0.1 하면 → 살짝 오른쪽 위 대각선 방향, 계속 theta += 0.1 하면
        # 점점 위쪽으로 커브를 그리며 이동. 에이전트는 불확실한 환경에서도 잘 작동하도록 학습한다,

        self.state = np.array([x, y, theta, v], dtype=np.float32)  # 상태 업데이트

        # 보상 설계 : 중앙선 패널티(완만): 편차 완화. 이런 보상 설계는 중앙산 근처로 주행을 유도하기 위한 것임
        center_penalty = -0.05 * abs(y - 50.00)  # y축 중앙선(50)에서 멀어질수록 감점, 도로 중앙 y=50을 기준으로 변경

        # 진행 보상: 앞으로 간 만큼 보상 (뒤로/옆걸음 억제)
        progress = max(0.0, x - x_prev) * 0.8
        alive = 0.2   # 생존 보상은 작게 (항상 +1.0은 과함)

        reward = alive + center_penalty + progress

        # 종료 조건
        terminated = False
        truncated = False

        if x > 90 and 0 <= y <= 100:    # 목적 달성인 경우 최고 보상 후 종료
            reward += 50.0
            terminated = True
        elif not (0 <= x <= 100 and 0 <= y <= 100):
            reward -= 15.0
            terminated = True

        return self.state, float(reward), terminated, truncated, {}
        # 반환 형식은 Gymnasium 표준: (next_state, reward, done, truncated, info)
        # truncated=False: 시간제한 종료가 아님, info={}: 추가 정보 없음

# 요약 :위 환경은 다음을 유도한다.
# x 방향으로 이동하여 90 이상 도달하면 성공, 도로 중앙선(y=50)을 유지하면서 이동할수록 더 많은 보상.
# 도로를 이탈하거나 목적지 도달 시 에피소드 종료.


In [ ]:
# DQN 모델생성: state를 입력받아, 각 action에 대한 Q-value를 출력하는 신경망 모델
# input_dim: state의 차원 수 ([x, y, θ, v] → 4차원), output_dim: action의 개수 (3개)
def create_dqn(input_dim, output_dim):
    model = keras.Sequential([
        keras.layers.Input(shape=(input_dim,)),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(output_dim, activation='linear')  #  Q-value는 실수값이므로
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')
    return model

episodes = 100
runs = 1    # 전체 학습실험을 몇 번 반복할지 설정. 보통 3회 정도 줌. 강화학습은 무작위 알고리즘.
            # 한 번만 학습해서 나온 결과는 우연일 수 있다. 평균 성능을 봐야 신뢰할 수 있는 평가됨

# 실험 결과 저장 변수 선언
all_run_rewards = [ ]    # 에피소드 별 총 보상을 저장.
# 예를 들어 runs = 3일 때 all_run_rewards → 길이 3 리스트, 각각 100개의 보상 기록 (에피소드 수만큼)

all_run_deviations = [ ]  # 도로 중앙(y=50)으로부터의 평균 편차 저장
final_trajectories = [ ]  # 에피소드별 이동 경로(trajectory)를 저장. 애니메이션이나 시각화에 사용할 계획


In [ ]:
for run in range(runs):
    env = MiniCarEnv()
    state_dim = env.observation_space.shape[0]
    num_actions = env.action_space.n            # 가능한 행동의 개수 = 3
    model = create_dqn(state_dim, num_actions)
    target_model = create_dqn(state_dim, num_actions)
    target_model.set_weights(model.get_weights())  # 두 모델을 동일하게 초기화
    # 두 모델을 쓰는 이유는 Q값의 불안정한 업데이트를 안정시키기 위해서.
    # model은 계속 업데이트, target_model은 비교 기준
    # DQN에서는 타겟모델을 일정 주기마다 업데이트해서 학습을 더안정적으로 만들기 위해 사용

    # 하이퍼파라미터 설정과 학습 결과 저장 변수 초기화
    gamma = 0.99     # discount factor
    epsilon = 0.6
    epsilon_min = 0.02
    epsilon_decay = 0.997  # 매 에피소드 후 epsilon *= 0.997로 점차 줄임
    # 에이전트는 초기에는 아무거나 해보고, 점차 학습한 정책을 따르게 됨
    # → 초반 탐험은 충분히, 중후반엔 탐험이 더 빨리 진정.

    batch_size = 32  # 미니배치 학습. 경험을 32개씩 샘플링해서 학습. (보통 32~64 사용)
    memory = deque(maxlen=2000)  # Replay Buffer. 최대 2000개의 경험을 저장
    # (state, action, reward, next_state, done) 튜플 저장.

    # 학습 결과 저장용 변수
    reward_history = []     # 에피소드마다 받은 총 보상을 저장 (성공/실패 여부 확인)
    deviation_history = []  # 주행 중 y값이 도로 중심(50)에서 얼마나 벗어났는지 평균편차 저장
    run_trajectories = []   # 미니카가 이동한 좌표들 기록 (시각화나 애니메이션용)
    CENTER_Y = 50.0         # 도로의 중앙선을 기준으로 편차 계산에 사용

    # 에이전트의 학습 루프에서 한 개의 에피소드를 실행하기 위한 준비 단계
    for ep in range(episodes):
        # 한 에피소드는 시작부터 종료 조건(done == True)에 도달할 때까지 1회 주행을 의미
        state, _ = env.reset()  # 환경을 초기화하고 시작 상태(state)를 받아옴. info는 개무시
        print(f"[Run {run}] 에피소드 시작 : ep={ep}, 초기 상태={state}")

        total_reward = 0   # 에피소드에서 받은 총 보상을 누적할 변수
        trajectory = []    # 미니카가 지나온 상태들을 기록 (좌표 시각화 등 용도)
        done = False       # 이 플래그가 True가 되면 에피소드 종료 (목표 도달 or 도로 이탈)

        # 하나의 에피소드 내에서 에이전트가 최대 200번까지 행동하는 과정을 의미하며,
        # 각 스텝마다 상태를 바탕으로 행동을 선택하고, 환경의 피드백을 받고, 경험을 저장
        for step in range(200):
            state_input = np.reshape(state, [1, state_dim])  # 1차원 -> 2차원

            # 이 if문이 policy. state에 따라 action을 할지 결정
            if np.random.rand() < epsilon:
                action = np.random.choice(num_actions)
            else:
                q_values = model.predict(state_input, verbose=0)
                action = np.argmax(q_values[0])
                # exploitation. 가장 Q값이 높은 행동선택 → 정책기반 행동선택

            next_state, reward, done, _, _ = env.step(action)  # 선택한 action을 환경에 전달
            memory.append((state, action, reward, next_state, done))  # 경험 저장 (replay buffer에 저장).  어떤 state에서 어떤 action을 했고, 어떤 reward를 받고, 어떤 next_state로 바뀌었는지, 에피소드 끝?(done)

            # 경로 및 보상 누적
            trajectory.append(state)  # 이동한 좌표들 저장 (애니메이션/시각화용)
            total_reward += reward
            state = next_state
            if done:  # 즉시 종료(도로이탈, 목표지점(x > 90)에 도달, 그외 종료조건 충족된 경우)
                break

        # 하나의 에피소드가 끝난 뒤, 그 동안의 성과(보상, 이동경로, 편차 등)를 기록하는 부분
        reward_history.append(total_reward)  # 학습성과 추이를 그래프로 보거나 비교시 사용
        traj = np.array(trajectory)  # 이동경로→NumPy 배열로 변환(벡터연산(예:편차계산) 가능)
        mean_deviation = np.mean(np.abs(traj[:, 1] - CENTER_Y))  # 평균 편차 계산(도로 중심에서 얼마나 안 벗어고 주행했는지).    traj[:, 1]→ trajectory에서 y좌표만 추출, np.abs(... - CENTER_Y) → 도로 중심선(y=50)으로부터의 편차 계산
          # np.mean(...) → 평균 편차 = 얼마나 중앙선을 유지했는지 평가하는 지표

        deviation_history.append(mean_deviation)  # 각 에피소드의 평균 편차를 저장
        # 그래프 작성, 모델이 점점 도로 중심을 잘 따라가는지 확인할 때 사용하기 위함

        run_trajectories.append((ep, trajectory.copy())) # 이동경로저장(시각화용).copy()는 고정본 저장
        print(f"[Run {run}] 에피소드 종료 : ep={ep}, 총 보상={total_reward:.2f}, \
                            편차={mean_deviation:.2f}, 종료 상태={state}")

        # 메모리에서 무작위 샘플링 후 Q-network 업데이트
        if len(memory) >= batch_size:  # 너무 적은 데이터로 학습하면 불안정해지기 때문에 일정량 이상 쌓이면 학습
            minibatch = random.sample(memory, batch_size)  # 리플레이 버퍼에서 무작위로 32개 샘플 추출.  무작위 추출을 통해 샘플의 상관관계를 줄이고 안정적인 학습을 유도
            states, targets = [], []  # 학습에 사용할 states, targets 누적 리스트

            # 각 샘플에 대해 Q값 계산 및 타겟 계산
            # 각 튜플의 구성: (현재상태, 행동, 보상, 다음상태, 종료여부)
            for s, a, r, s_next, d in minibatch:
                # 상태 형태 변환. Keras모델 입력형태로 reshape →(1, 4)
                s_input = np.reshape(s, [1, state_dim])
                s_next_input = np.reshape(s_next, [1, state_dim])

                target = model.predict(s_input, verbose=0)[0]
                # 현재 상태 s에서 각 행동의 Q값을 예측.  예: [0.5, 1.2, 0.3] → 행동 0,1,2에 대한 Q값

                # 타겟 Q값 계산
                if d:       # done = True → 다음 상태가 없으므로 Q(s, a) ← r 그대로 사용
                    target[a] = r   # 종료된 상태면: 단순히 그 보상이 정답
                else:       # done = False → 타겟 네트워크로 다음 상태의 Q값 계산
                    t_next = target_model.predict(s_next_input, verbose=0)[0]
                    target[a] = r + gamma * np.max(t_next)   # 벨만 식. 정책 개선
                # 학습 입력/타겟 누적
                states.append(s)       # s: 입력 상태
                targets.append(target) # target: 해당 상태에서의 예측 Q값 중 하나만 수정된 벡터

            # Q-network (model)를 학습하는 부분.
            # states는 입력값, targets는 그에 맞는 정답(Q-value 벡터).
            # np.array(states)는 (batch_size, 4) 크기의 2차원 배열.
            # np.array(targets)는 (batch_size, 3) 크기의 배열
            # → 각 행동(좌,직진,우)에 대한 Q-value
            model.fit(np.array(states), np.array(targets), epochs=1, verbose=0)

        # 탐험 확률 감소 (ε 줄이기).
        # 매 에피소드마다 epsilon *= 0.995 → 점점 줄어듦. 최소값(epsilon_min = 0.01)까지 감소함
        if epsilon > epsilon_min:
            epsilon *= epsilon_decay  # 학습이 진행될수록 무작위성이 줄고, 정책기반행동으로 수렴

        # 현재 Q-network (model)의 파라미터를 target_model에 복사.
        if ep % 10 == 0:   # 타겟 네트워크 주기적 갱신
            target_model.set_weights(model.get_weights())

    # 학습 실험(run)이 끝났을 때, 그 결과를 저장하거나 시각화용 데이터를 준비
    # reward_history: 하나의 run 안에서 각 에피소드별 총 보상 리스트.
    # 예: [3.2, 5.1, 8.0, ..., 12.5] ← 에피소드별 총 리워드 100개
    all_run_rewards.append(reward_history)   # 여러 run의 성능을 비교하고 평균을 낼 수 있게 됨
    # 각 run의 중심선 편차 기록 저장. 도로 중앙을 얼마나 잘 유지했는지 여러 run에 대해 비교 가능
    all_run_deviations.append(deviation_history)

    # run은 현재 실행 중인 학습 반복횟수 인덱스.
    # runs는 전체 학습을 몇 번 반복할지를 정한 값 (예:runs=3).
    # runs - 1은 마지막 run의 인덱스를 의미한다.
    if run == runs - 1:  # 마지막 run의 경로 데이터 저장. 마지막 run의 경로 데이터 저장
        final_trajectories = run_trajectories  # 시각화용


In [ ]:
# “성공률” 체크하는 간단 지표 : 성공 여부는 "ep 끝에서 x>90"로 판정
# 이 그래프가 우상향하고, 성공률이 70%+로 올라가면 “안정”으로 봐도 좋다.
success_flags = [traj_end[0] > 90 for _, traj in run_trajectories for traj_end in [traj[-1]]]
success_rate = sum(success_flags) / len(success_flags)
print(f"Success rate: {success_rate*100:.1f}%")

# 이동 평균 보상(예: 20-ep window)
def moving_avg(a, w=20):
    return np.convolve(a, np.ones(w)/w, mode='valid')

w = 20
ma = moving_avg(reward_history, w)
episodes = np.arange(w-1, w-1 + len(ma))  # 에피소드 번호 정렬

plt.figure(figsize=(6,3))
plt.plot(episodes, ma, label=f"Moving Avg (w={w})")
plt.title("Moving Avg Reward (window=20)")
plt.xlabel("Episode")  
plt.ylabel("Reward (moving average)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

"""
이 그래프는 매우 만족스러운 학습 결과를 보여주는 전형적인 DQN 패턴이다.
"""


# 편차(Deviation) 이동평균 그래프 → 중앙선을 얼마나 잘 유지하는지 시각적으로 확인 가능.
plt.plot(moving_avg(deviation_history, 20))
plt.title("Moving Avg Deviation (lower=better)")
plt.xlabel("Episode")
plt.ylabel("Mean deviation (|y-50|)")
plt.show()